# 7.14 Embeddings and Text Representation

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook explains how discrete text tokens become dense vectors that neural networks can learn from directly.


## 7.14.1 One-hot versus embeddings

### Why It Matters
One-hot vectors are sparse and high-dimensional, while embeddings are dense and learned. A tiny vocabulary is enough to compare the two representations directly.


In [ ]:
import torch
import torch.nn.functional as F

vocab_size = 6
token_id = torch.tensor([3])
one_hot = F.one_hot(token_id, num_classes=vocab_size).float()
embed = torch.nn.Embedding(vocab_size, 3)
dense = embed(token_id)
{"one_hot_shape": list(one_hot.shape), "embedding_shape": list(dense.shape), "embedding_vector": dense.squeeze(0).detach().round(decimals=3).tolist()}


### Interpretation
Embeddings shrink the representation while allowing the model to learn which tokens should be similar.


## 7.14.2 Learned embedding spaces

### Why It Matters
Embedding vectors are useful because they are trainable. Two tokens used in similar predictive contexts can move closer together during training.


In [ ]:
import torch
from torch import nn

torch.manual_seed(38)
pairs = torch.tensor([[0, 1], [0, 2], [3, 4], [3, 5]], dtype=torch.long)
labels = torch.tensor([1., 1., 1., 1.]).unsqueeze(1)
embed = nn.Embedding(6, 4)
scorer = nn.Linear(8, 1)
opt = torch.optim.Adam(list(embed.parameters()) + list(scorer.parameters()), lr=0.05)
loss_fn = nn.BCEWithLogitsLoss()
for _ in range(100):
    left = embed(pairs[:, 0])
    right = embed(pairs[:, 1])
    logits = scorer(torch.cat([left, right], dim=1))
    loss = loss_fn(logits, labels)
    opt.zero_grad()
    loss.backward()
    opt.step()
embed.weight.detach().round(decimals=3)


### Interpretation
The vectors change because the task pushes the space into a more useful geometry.


## 7.14.3 Word indices and padding

### Why It Matters
Real batches contain sequences of different lengths. Padding ensures they can share a rectangular tensor shape.


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

seqs = [torch.tensor([2, 4, 5]), torch.tensor([3, 1]), torch.tensor([4])]
padded = pad_sequence(seqs, batch_first=True, padding_value=0)
{"padded_sequences": padded.tolist(), "shape": list(padded.shape)}


### Interpretation
Padding tokens are part of the representation pipeline, not meaningful language on their own.


## 7.14.4 Sequence batching

### Why It Matters
Batched embeddings convert a matrix of token ids into a three-dimensional tensor: batch, time, and embedding dimension.


In [ ]:
import torch
from torch import nn

batch = torch.tensor([[1, 2, 3, 0], [4, 5, 0, 0]])
embed = nn.Embedding(6, 5)
out = embed(batch)
{"input_shape": list(batch.shape), "embedded_shape": list(out.shape)}


### Interpretation
That three-dimensional tensor is the standard input format for recurrent and transformer-style text models.


## 7.14.5 Similarity intuition

### Why It Matters
Cosine similarity is a simple way to ask whether two embeddings point in similar directions.


In [ ]:
import torch
import torch.nn.functional as F

a = torch.tensor([1.0, 1.0, 0.0])
b = torch.tensor([0.9, 1.1, 0.0])
c = torch.tensor([-1.0, 0.0, 0.0])
{"similarity_a_b": round(float(F.cosine_similarity(a, b, dim=0).item()), 3), "similarity_a_c": round(float(F.cosine_similarity(a, c, dim=0).item()), 3)}


### Interpretation
Similarity is what makes embedding spaces useful for clustering, retrieval, and downstream modeling.


## 7.14.6 Train a simple embedding-based text model

### Why It Matters
This final subsection fits a compact sentiment-style classifier using embeddings plus average pooling.


In [ ]:
import torch
from torch import nn

vocab = {"<pad>": 0, "good": 1, "bad": 2, "great": 3, "awful": 4, "movie": 5, "plot": 6}
texts = [[1, 3, 5], [2, 4, 6], [3, 5, 0], [2, 6, 0], [1, 5, 0], [4, 6, 0]]
labels = torch.tensor([1, 0, 1, 0, 1, 0], dtype=torch.long)
X = torch.tensor(texts, dtype=torch.long)

class AvgEmbedClassifier(nn.Module):
    def __init__(self, vocab_size, dim, classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=pad_idx)
        self.head = nn.Linear(dim, classes)
        self.pad_idx = pad_idx

    def forward(self, x):
        embedded = self.embedding(x)
        mask = (x != self.pad_idx).unsqueeze(-1)
        pooled = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.head(pooled)

model = AvgEmbedClassifier(len(vocab), 6, 2, pad_idx=vocab["<pad>"])
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()
for _ in range(120):
    logits = model(X)
    loss = loss_fn(logits, labels)
    opt.zero_grad()
    loss.backward()
    opt.step()
with torch.no_grad():
    preds = model(X).argmax(dim=1)

{
    "training_accuracy": round(float((preds == labels).float().mean().item()), 3),
    "final_loss": round(float(loss.item()), 4),
}


### Interpretation
The model is simple, but it already shows how embeddings turn raw token ids into learned task features.
